# Document Processing with Unstructured Transform MCP

In this recipe, we connect to [Unstructured Transform](https://docs.unstructured.io/transform/overview)'s hosted MCP server and use a Haystack `Agent` to parse and chunk a document, entirely through MCP tools.

**Services used:**
- [Unstructured Transform MCP](https://mcp.transform.unstructured.io): document processing (partition, enrich, chunk, embed) exposed as MCP tools
- [Anthropic Claude](https://www.anthropic.com/): LLM for agent reasoning

## Install dependencies

In [ ]:
!pip install -q haystack-ai mcp-haystack anthropic-haystack

## Set up API keys

You'll need two API keys:
- **Unstructured API key**: get one from the [Transform get-started page](https://transform.unstructured.io/get-started) after signing in. The free tier includes 15,000 pages a month.
- **Anthropic API key**: get one at [console.anthropic.com](https://console.anthropic.com/)

In [ ]:
import os
from getpass import getpass

if "UNSTRUCTURED_API_KEY" not in os.environ:
    os.environ["UNSTRUCTURED_API_KEY"] = getpass("Enter your Unstructured API key: ")
if "ANTHROPIC_API_KEY" not in os.environ:
    os.environ["ANTHROPIC_API_KEY"] = getpass("Enter your Anthropic API key: ")

## Step 1: Connect to Unstructured Transform MCP

[Unstructured Transform](https://docs.unstructured.io/transform/overview) exposes its document-processing pipeline (partition, enrich, chunk, embed) as a hosted MCP server at `https://mcp.transform.unstructured.io`. We connect to it with [`MCPToolset`](https://docs.haystack.deepset.ai/docs/mcptoolset), using `StreamableHttpServerInfo`'s native `token` parameter to send the Unstructured API key as an `Authorization: Bearer` header.

In [ ]:
from haystack_integrations.tools.mcp import MCPToolset, StreamableHttpServerInfo
from haystack.utils import Secret

server_info = StreamableHttpServerInfo(
    url="https://mcp.transform.unstructured.io",
    token=Secret.from_env_var("UNSTRUCTURED_API_KEY"),
)
toolset = MCPToolset(server_info=server_info, eager_connect=True)

for tool in toolset.tools:
    print(f"{tool.name}: {tool.description}")

The pipeline runs asynchronously as a job: submit a file for processing, poll until it's done, then fetch the rendered result; a separate helper mints an upload URL for files that aren't already reachable over HTTPS. Unstructured adds tools and capabilities to this server as they ship new features, so rather than list exact tool names and a fixed count here, the cell above discovered the live toolset at connect time, and the agent below matches tools to each step by description.

## Step 2: Build a Haystack Agent with the Transform MCP toolset

We give the agent the toolset directly, along with a system prompt describing the asynchronous submit -> poll -> fetch flow by behavior rather than by hardcoded tool name, since the agent needs to poll for a result rather than get one back immediately, and this way the prompt keeps working as Unstructured renames or adds tools.

In [ ]:
from haystack.components.agents import Agent
from haystack_integrations.components.generators.anthropic import AnthropicChatGenerator

agent = Agent(
    chat_generator=AnthropicChatGenerator(
        model="claude-opus-4-6",
        generation_kwargs={"max_tokens": 4096},
    ),
    tools=toolset,
    system_prompt="""You are a document-processing assistant with access to Unstructured Transform MCP tools. Check the tools available to you and use whichever ones match the steps below by description, since exact tool names may change over time.

Transform jobs are asynchronous. When asked to process a document:
1. Submit the file reference(s) and the requested processing stages to start a processing job. This returns a job ID immediately; the job itself runs in the background.
2. Check the job's status, waiting a few seconds between checks, until it reports as complete.
3. Fetch the job's rendered output using its job ID, and summarize it for the user.
""",
)

## Step 3: Process a document end-to-end

We hand the agent a publicly reachable PDF and ask it to parse and chunk it. No local file or upload step is needed here, since the job-submission tool accepts `https://` URLs directly.

In [ ]:
from haystack.dataclasses import ChatMessage

pdf_url = "https://arxiv.org/pdf/1706.03762"

result = agent.run(
    messages=[
        ChatMessage.from_user(
            f"Parse and chunk the PDF at {pdf_url}. "
            "Use the 'hi_res' partition strategy, and chunk with chunk_by_title, "
            "max_characters=1000. Once the job is complete, fetch the results as "
            "markdown and show me the first two chunks."
        )
    ]
)

In [ ]:
print(result["last_message"].text)

## Conclusion

Unstructured Transform's MCP server brings partitioning, enrichment, chunking, and embedding into a single set of tools an agent can call directly, without wiring up a separate ETL pipeline. Because job submission runs asynchronously and returns a job ID right away, a Haystack `Agent` can poll for completion and fetch results the same way it would call any other tool, making it straightforward to drop document processing into a larger agentic workflow.